# Using the Model Output

This notebook is just a quick check of the evaluation output to show basic Truth/Reconstructed histograms of predicted quantities. This provides a starting point for using the model output for other analysis tasks.

In [ ]:
import uproot
import matplotlib.pyplot as plt
import numpy as np
import awkward as ak
from parnassus_core.utils.pid import pid_onehot_ids

eval_output = "/global/homes/t/tjhellst/link_to_work/parnassus-core/clasdis_filtered/clasdis_filtered_eval_19k.root"
data = uproot.open(eval_output)["evt_tree"]
print(data.keys())

In [ ]:
# load data arrays
fast_sim = data["pflow"].array()
truth = data["truth"].array()

# apply the particle level mask with ind > 0 to remove buffer from the model output
fast_sim = fast_sim[fast_sim["ind"] > 0]
truth = truth[truth["ind"] > 0]

print("fast sim type:", fast_sim.type)
print("truth type:", truth.type)

In [ ]:
# plot styling:
var_styles = {
    "p": {"bins": np.linspace(0, 10, 100), "xlabel": "p (GeV)"},
    "theta": {"bins": np.linspace(0.1, 0.6, 100), "xlabel": "theta (rad)"},
    "phi": {"bins": np.linspace(-3.14, 3.14, 100), "xlabel": "phi (rad)"},
    "vx": {"bins": np.linspace(-0.1, 0.1, 100), "xlabel": "vx (cm)"},
    "vy": {"bins": np.linspace(-0.1, 0.1, 100), "xlabel": "vy (cm)"},
    "vz": {"bins": np.linspace(-0.1, 0.1, 100), "xlabel": "vz (cm)"},
    "pdgId": {"bins": np.arange(13) - 0.5, "xlabel": "Particle species"},
    "N": {"bins": np.linspace(0, 10, 11) - 0.5, "xlabel": "Multiplicity"},
}

pid_labels = [
    r"$\gamma$",
    r"$e^-$",
    r"$e^+$",
    r"$\pi^+$",
    r"$\pi^-$",
    r"$K^+$",
    r"$K^-$",
    r"$p$",
    r"$\bar{p}$",
    r"$n$",
    r"$d$",
    "other",
]

In [ ]:
def pid_index(collection):
    # map pdgIds to species bins; anything not in pid_onehot_ids goes to "other"
    pids = np.rint(ak.to_numpy(ak.flatten(collection["pdgId"]))).astype(int)
    idx = np.full(len(pids), len(pid_onehot_ids))
    for i, pid in enumerate(pid_onehot_ids):
        idx[pids == pid] = i
    return idx


# plot the truth and fast sim distributions for comparison
def plot_histogram(fast_sim, truth, var, ax, density=True):
    if var == "N":
        # for multiplicity, we want to count the number of particles per event
        fast_sim_counts = ak.num(fast_sim)
        truth_counts = ak.num(truth)
        ax.hist(
            fast_sim_counts,
            bins=var_styles[var]["bins"],
            label="Fast Sim",
            color="#1f77b4",
            alpha=0.5,
            histtype="stepfilled",
            density=density,
        )
        ax.hist(
            truth_counts,
            bins=var_styles[var]["bins"],
            label="Truth",
            color="black",
            histtype="step",
            density=density,
        )
    elif var == "pdgId":
        # for pdgId, we want the total yield per particle species, so never normalize
        density = False
        ax.hist(
            pid_index(truth),
            bins=var_styles[var]["bins"],
            label="Truth",
            color="black",
            histtype="step",
        )
        ax.hist(
            pid_index(fast_sim),
            bins=var_styles[var]["bins"],
            label="Fast Sim",
            color="#1f77b4",
            alpha=0.5,
            histtype="stepfilled",
        )
        ax.set_xticks(np.arange(len(pid_labels)))
        ax.set_xticklabels(pid_labels)
        ax.set_yscale("log")
    else:
        ax.hist(
            ak.flatten(truth[var]),
            bins=var_styles[var]["bins"],
            label="Truth",
            color="black",
            histtype="step",
            density=density,
        )
        ax.hist(
            ak.flatten(fast_sim[var]),
            bins=var_styles[var]["bins"],
            label="Fast Sim",
            color="#1f77b4",
            alpha=0.5,
            histtype="stepfilled",
            density=density,
        )

    ax.set_xlabel(var_styles[var]["xlabel"])
    ylabel = "Normalized Counts" if density else "Counts"
    ax.set_ylabel(ylabel)
    ax.set_title(f"{var} Distribution")
    ax.legend()


fig, axs = plt.subplots(3, 3, figsize=(20, 20))
axs_flat = axs.flatten()
for i, var in enumerate(var_styles.keys()):
    plot_histogram(fast_sim, truth, var, axs_flat[i])